In [ ]:
!pip install wandb gensim sentence-transformers transformers sentencepiece faiss-cpu -q

import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import os
os.environ["WANDB_SILENT"] = "true"

from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
w_key = user_secrets.get_secret("WANDB_Key")
wandb.login(key=w_key)

import torch
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel, T5ForConditionalGeneration, pipeline

OPTS   = ['A', 'B', 'C', 'D', 'E']
device = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("train:", train.shape, "| test:", test.shape, "| device:", device)
train.head(3)

In [ ]:
print("=== null counts ===")
print(train.isnull().sum())

print("\n=== answer distribution ===")
print(train['answer'].value_counts())

print("\n=== prompt length (words) ===")
print(train['prompt'].str.split().str.len().describe().round(2))

print("\n=== avg option length (words) ===")
for o in OPTS:
    print(f"  {o}: {train[o].str.split().str.len().mean():.1f}")

In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

def remove_stopwords(tokens):
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS and len(t) > 1]

for col in ['prompt'] + OPTS:
    train[f'clean_{col}'] = train[col].apply(clean_text)
    test[f'clean_{col}']  = test[col].apply(clean_text)

# vocab size of cleaned prompts
all_words = []
for p in train['clean_prompt']:
    all_words.extend(p.split())

print(f"unique vocab in cleaned prompts: {len(set(all_words))}")
print(f"sample — original : {train['prompt'].iloc[0]}")
print(f"sample — cleaned  : {train['clean_prompt'].iloc[0]}")

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, util as st_util
import faiss

hf_train = Dataset.from_pandas(train)
hf_test  = Dataset.from_pandas(test)
print(hf_train)

# tfidf on full corpus so test vocab is covered
full_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        full_corpus.append(row['prompt'])
        for o in OPTS:
            full_corpus.append(row[o])
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), max_features=25000, sublinear_tf=True)
tfidf.fit(full_corpus)
print(f"tfidf vocab: {len(tfidf.vocabulary_)}")

# word2vec on cleaned stopword-removed tokens
w2v_corpus = []
for df in [train, test]:
    for _, row in df.iterrows():
        w2v_corpus.append(remove_stopwords(tokenize(row['prompt'])))
        for o in OPTS:
            w2v_corpus.append(remove_stopwords(tokenize(row[o])))
w2v_model = Word2Vec(sentences=w2v_corpus, vector_size=150, window=7, min_count=2, workers=4, sg=1, epochs=15, seed=42)
print(f"w2v vocab: {len(w2v_model.wv)}")

# roberta for context aware embeddings and attention inspection
bert_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
bert_model     = AutoModel.from_pretrained("roberta-base", output_attentions=True).to(device)
bert_model.eval()
sample_inputs = bert_tokenizer(train['prompt'].iloc[0], return_tensors="pt").to(device)
with torch.no_grad():
    sample_out = bert_model(**sample_inputs)
print(f"\nattention layers: {len(sample_out.attentions)}, heads per layer: {sample_out.attentions[0].shape[1]}")
print("this is how roberta learns which words to focus on relative to others")

# zero shot nli pipeline deberta is stronger than bart for entailment tasks
zero_shot = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-large",
    device=0 if device == "cuda" else -1
)
print("zero-shot pipeline loaded: nli-deberta-v3-large")

# minilm for building the retrieval vector database
# fast and good enough for dense retrieval
minilm  = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# knowledge base: prompt + correct answer so retrieved docs give useful signal
kb_docs = [f"{row['prompt']} Answer: {row[row['answer']]}" for _, row in train.iterrows()]
kb_embs = minilm.encode(kb_docs, batch_size=32, show_progress_bar=True).astype(np.float32)
faiss.normalize_L2(kb_embs)

# IndexFlatIP = exact inner product search, equivalent to cosine sim on normalized vecs
faiss_index = faiss.IndexFlatIP(kb_embs.shape[1])
faiss_index.add(kb_embs)
print(f"faiss index built: {faiss_index.ntotal} documents indexed")

In [ ]:
def tfidf_cosine(text1, text2):
    v1 = tfidf.transform([text1])
    v2 = tfidf.transform([text2])
    return float(cosine_similarity(v1, v2)[0][0])

def avg_w2v(text):
    tokens = remove_stopwords(tokenize(text))
    vecs   = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v_model.vector_size)

def w2v_cosine(text1, text2):
    v1 = avg_w2v(text1).reshape(1, -1)
    v2 = avg_w2v(text2).reshape(1, -1)
    if np.all(v1 == 0) or np.all(v2 == 0):
        return 0.0
    return float(cosine_similarity(v1, v2)[0][0])

# roberta mean pooling ignores padding tokens via attention mask
def get_roberta_embedding(text):
    inputs     = bert_tokenizer(text, return_tensors="pt", truncation=True, max_length=64, padding=True).to(device)
    with torch.no_grad():
        out    = bert_model(**inputs)
    token_embs = out.last_hidden_state
    mask       = inputs['attention_mask'].unsqueeze(-1).expand(token_embs.size()).float()
    summed     = torch.sum(token_embs * mask, dim=1)
    counts     = torch.clamp(mask.sum(dim=1), min=1e-9)
    return (summed / counts).squeeze(0).cpu().numpy()

def rank_tfidf(row):
    sims = {o: tfidf_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_w2v(row):
    sims = {o: w2v_cosine(row['prompt'], row[o]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

def rank_roberta(row):
    pv   = get_roberta_embedding(row['prompt']).reshape(1, -1)
    sims = {o: float(cosine_similarity(pv, get_roberta_embedding(row[o]).reshape(1, -1))[0][0]) for o in OPTS}
    return sorted(sims, key=lambda x: sims[x], reverse=True)

# shared helper returns raw scores dict so both zeroshot and rag can use it
# framing as hypothesis improves NLI model accuracy over raw option text
def zeroshot_scores(row, prompt_override=None):
    prompt           = prompt_override if prompt_override else row['prompt']
    candidate_labels = [f"The correct answer is: {row[o]}" for o in OPTS]
    label_to_letter  = {f"The correct answer is: {row[o]}": o for o in OPTS}
    result           = zero_shot(prompt, candidate_labels, multi_label=True)
    return {label_to_letter[l]: s for l, s in zip(result['labels'], result['scores'])}

def rank_zeroshot(row):
    scores = zeroshot_scores(row)
    return sorted(scores, key=lambda x: scores[x], reverse=True)

# retrieve similar training examples from faiss and prepend as context
# exclude_self=True when evaluating on train so the question itslf isnt retrieved
def retrieve_context(query, k=3, exclude_self=False):
    q_emb = minilm.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q_emb)
    n = k + 1 if exclude_self else k
    _, indices = faiss_index.search(q_emb, n)
    idxs = indices[0][1:] if exclude_self else indices[0]
    return [kb_docs[i] for i in idxs]

def rank_rag(row, exclude_self=False):
    contexts  = retrieve_context(row['prompt'], k=3, exclude_self=exclude_self)
    augmented = " | ".join(contexts) + " Now answer this: " + row['prompt']
    scores    = zeroshot_scores(row, prompt_override=augmented)
    return sorted(scores, key=lambda x: scores[x], reverse=True)

def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted):
        if p == actual:
            hits  += 1
            score += hits / (i + 1.0)
    return score

def mapk(actuals, predictions, k=3):
    return sum(apk(a, p, k) for a, p in zip(actuals, predictions)) / len(actuals)

actuals        = train['answer'].tolist()
top3_maj       = list(train['answer'].value_counts().index[:3])
map3_maj       = round(mapk(actuals, [top3_maj] * len(train)), 4)
tfidf_preds    = [rank_tfidf(row)[:3] for _, row in train.iterrows()]
map3_tfidf     = round(mapk(actuals, tfidf_preds), 4)
w2v_preds      = [rank_w2v(row)[:3] for _, row in train.iterrows()]
map3_w2v       = round(mapk(actuals, w2v_preds), 4)
sample_roberta = train.sample(n=200, random_state=42)
roberta_preds  = [rank_roberta(row)[:3] for _, row in sample_roberta.iterrows()]
map3_roberta   = round(mapk(sample_roberta['answer'].tolist(), roberta_preds), 4)

print("running zero-shot on full train...")
zs_preds  = [rank_zeroshot(row)[:3] for _, row in train.iterrows()]
map3_zs   = round(mapk(actuals, zs_preds), 4)

print("running RAG on full train...")
rag_preds = [rank_rag(row, exclude_self=True)[:3] for _, row in train.iterrows()]
map3_rag  = round(mapk(actuals, rag_preds), 4)

print(f"\nmajority baseline    MAP@3 : {map3_maj}")
print(f"tfidf cosine         MAP@3 : {map3_tfidf}")
print(f"word2vec cosine      MAP@3 : {map3_w2v}")
print(f"roberta cosine (200) MAP@3 : {map3_roberta}")
print(f"zero-shot nli        MAP@3 : {map3_zs}")
print(f"rag + zero-shot      MAP@3 : {map3_rag}")

run = wandb.init(
    project = "23f1000362-t22026",
    name    = "milestone3-rag-pipeline",
    config  = {
        "milestone": 3, "retriever": "all-MiniLM-L6-v2",
        "vector_db": "faiss-flatIP", "kb_size": len(kb_docs),
        "ranker": "cross-encoder/nli-deberta-v3-large", "top_k_retrieval": 3
    },
    tags = ["milestone-3", "rag"]
)
wandb.log({
    "map3/majority_baseline"  : map3_maj,
    "map3/tfidf_cosine"       : map3_tfidf,
    "map3/word2vec_cosine"    : map3_w2v,
    "map3/roberta_cosine_200" : map3_roberta,
    "map3/zeroshot_nli"       : map3_zs,
    "map3/rag_zeroshot"       : map3_rag
})
wandb.finish()

best_model = "rag" if map3_rag > map3_zs else "zeroshot"
print(f"\nbest model on train: {best_model}")

test_preds = []
for _, row in test.iterrows():
    ranked = rank_rag(row, exclude_self=False) if best_model == "rag" else rank_zeroshot(row)
    test_preds.append(" ".join(ranked[:3]))

submission = pd.DataFrame({'ID': test['id'].values, 'Prediction': test_preds})
submission.to_csv("submission.csv", index=False)
print("submission.csv saved")
submission.head(5)

In [ ]:
#Milestone 3
!pip install faiss-cpu -q

from sentence_transformers import CrossEncoder

train_iitm = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

kb = []
for idx, row in train_iitm.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

iitm_model    = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = iitm_model.encode(kb, show_progress_bar=False)
iitm_index    = faiss.IndexFlatL2(kb_embeddings.shape[1])
iitm_index.add(kb_embeddings)

zs            = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0 if device=="cuda" else -1)
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

row_150    = train_iitm.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150[o]) for o in ['A','B','C','D','E']]
ans_150    = str(row_150[row_150['answer']])

print(f"kb size: {len(kb)}")
print(f"iitm index: {iitm_index.ntotal} docs")
print(f"row 150 correct answer: {row_150['answer']} = {ans_150[:60]}")